## DX702 Coding Homework ##
Keshia-Lee Martin<br>
DX702 Experimental Design and Causality<br>
October 11, 2026

### Week 1

In [115]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
from sklearn.neighbors import NearestNeighbors
from scipy.stats import skew

hw1_1 = pd.read_csv("homework_1.1.csv")
hw1_2 = pd.read_csv("homework_1.2.csv")

Question 1: Which of the following is closest to the coefficient of X1?

In [116]:
model = smf.ols("Y ~ X1 + X2 + X3", data=hw1_1).fit()
coeff_x1 = model.params["X1"]
print(round(coeff_x1, 3))

1.007


Question 2: Which Xi has the greatest difference between the amount Y increases for each 1 unit of Xi (fixing the other Xi's), as opposed to the amount that Y increases for each 1 unit of Xi in the dataset on average (not fixing the other Xi's)?
Hint: For the former, regress Y on Xi alone. For the latter, regress Y on all three Xi's.


In [117]:
only_x1   = smf.ols("Y ~ X1", data=hw1_1).fit()
only_x2   = smf.ols("Y ~ X2", data=hw1_1).fit()
only_x3   = smf.ols("Y ~ X3", data=hw1_1).fit()

coeff_x1 = model.params["X1"]
coeff_x2 = model.params["X2"]
coeff_x3 = model.params["X3"]

coeff_x1_alone = only_x1.params["X1"]
coeff_x2_alone = only_x2.params["X2"]
coeff_x3_alone = only_x3.params["X3"]

print("X1:", round(coeff_x1, 3), " alone", round(coeff_x1_alone, 3),
      " diff", round(coeff_x1_alone - coeff_x1, 3))
print("X2:", round(coeff_x2, 3), " alone", round(coeff_x2_alone, 3),
      " diff", round(coeff_x2_alone - coeff_x2, 3))
print("X3:", round(coeff_x3, 3), " alone", round(coeff_x3_alone, 3),
      " diff", round(coeff_x3_alone - coeff_x3, 3))

X1: 1.007  alone 1.842  diff 0.835
X2: 1.965  alone 4.084  diff 2.119
X3: 2.975  alone 3.097  diff 0.122


Question 3: When regressing Y on all Xi's together, which coefficient is most significant, taking the t-statistic as the measure of significance?

In [118]:
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      Y   R-squared:                       0.991
Model:                            OLS   Adj. R-squared:                  0.991
Method:                 Least Squares   F-statistic:                 3.543e+04
Date:                Sun, 11 Oct 2026   Prob (F-statistic):               0.00
Time:                        21:10:46   Log-Likelihood:                -727.62
No. Observations:                1000   AIC:                             1463.
Df Residuals:                     996   BIC:                             1483.
Df Model:                           3                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      0.0026      0.016      0.166      0.8

Question 4: What is the distance of the farthest match in this set?

In [119]:
treated = hw1_2[hw1_2["X"] == 1].reset_index(drop=True)
control = hw1_2[hw1_2["X"] == 0].reset_index(drop=True)

n_neighbor = NearestNeighbors(n_neighbors=1).fit(control[["Z"]])
distances, positions = n_neighbor.kneighbors(treated[["Z"]])
 
print("Treated units =", len(treated))
print("Farthest match distance =", distances.max())


Treated units = 48
Farthest match distance = 0.2102170871093757


Question 5: What is the effect? The difference between the average Y for the X = 0 values and the average Y for the X = 1 values, where the X = 0 sample is the set of best matches — so the matched sample of X = 0 against the full sample of X = 1.

In [120]:
matched = control.iloc[positions.flatten()]

print("mean Y treated =", treated["Y"].mean())
print("mean Y matched control =", matched["Y"].mean())
print("Effect =", treated["Y"].mean() - matched["Y"].mean())


mean Y treated = 1.125597137873841
mean Y matched control = 0.582237072655257
Effect = 0.5433600652185839


Question 6: How many duplicates do you end up with? Count all but the first in each group. One way to do this is radius_neighbors.

In [121]:
pairs_total = 0
controls_used = set()

n_neighbor2 = NearestNeighbors(radius=0.2).fit(control[["Z"]])
distances, groups = n_neighbor2.radius_neighbors(treated[["Z"]])

for group in groups:
    pairs_total = pairs_total + len(group)
    for position in group:
        controls_used.add(int(position))

print(pairs_total)
print(len(controls_used))
print(pairs_total - len(controls_used))

737
52
685


Question 7: What is the effect?


In [122]:
averages = []
for group in groups:
    if len(group) > 0:
        averages.append(control.iloc[group]["Y"].mean())
 
overall_control = sum(averages) / len(averages)
 
print(len(averages))
print(overall_control)
print(treated["Y"].mean() - overall_control)


46
0.5411846604492228
0.5844124774246182


### Week 2

Which of these is closest to being the coefficient of group 1? 

In [123]:
hw2_1 = pd.read_csv("homework_2.1.csv")
hw2_2 = pd.read_csv("homework_2.2.csv")


In [124]:
model_g1 = smf.ols("G1 ~ time", data=hw2_1).fit()
model_g2 = smf.ols("G2 ~ time", data=hw2_1).fit()
model_g3 = smf.ols("G3 ~ time", data=hw2_1).fit()
 
slope_g1 = model_g1.params["time"]
slope_g2 = model_g2.params["time"]
slope_g3 = model_g3.params["time"]
 
print("G1 =", slope_g1)
print("G2 =", slope_g2)
print("G3 =", slope_g3)


G1 = 0.008498349168750096
G2 = 0.0070804680232676815
G3 = 0.011472822938047054


Which of these is closest to being the common linear coefficient for all groups?

In [125]:
g1_rows = hw2_1[["time", "G1"]].rename(columns={"G1": "y"})
g2_rows = hw2_1[["time", "G2"]].rename(columns={"G2": "y"})
g3_rows = hw2_1[["time", "G3"]].rename(columns={"G3": "y"})

g1_rows["group"] = "G1"
g2_rows["group"] = "G2"
g3_rows["group"] = "G3"

all_rows = pd.concat([g1_rows, g2_rows, g3_rows])

all_rows["is_g1"] = (all_rows["group"] == "G1").astype(int)
all_rows["is_g2"] = (all_rows["group"] == "G2").astype(int)
all_rows["is_g3"] = (all_rows["group"] == "G3").astype(int)

pooled = smf.ols("y ~ is_g1 + is_g2 + is_g3 + time - 1", data=all_rows).fit()

fixed_effect_g1 = pooled.params["is_g1"]
fixed_effect_g2 = pooled.params["is_g2"]
fixed_effect_g3 = pooled.params["is_g3"]
common_slope = pooled.params["time"]

print("G1 fixed effect =", fixed_effect_g1)
print("G2 fixed effect =", fixed_effect_g2)
print("G3 fixed effect =", fixed_effect_g3)
print("common slope, time =", common_slope)


G1 fixed effect = 0.07855194562167919
G2 fixed effect = 0.5896544232284033
G3 fixed effect = 0.26903180757269846
common slope, time = 0.009017213376688394


If we were to measure the effect of the treatment simply by subtracting the outcomes of the treated vs. untreated population, which of these is closest to the mean effect? (This is not the recommended way of measuring the mean effect when there are confounders!) 

In [126]:
treated_y = hw2_2[hw2_2["X"] == 1]["Y"]
control_y = hw2_2[hw2_2["X"] == 0]["Y"]
 
mean_treated = treated_y.mean()
mean_control = control_y.mean()
naive_effect = mean_treated - mean_control
 
print("n treated =", len(treated_y), " n control =", len(control_y))
print("mean Y, treated =", mean_treated)
print("mean Y, control =", mean_control)
print("naive effect =", naive_effect)


n treated = 1972  n control = 8028
mean Y, treated = 7.842756767343214
mean Y, control = 4.922039502620024
naive effect = 2.9207172647231907


If we were to use bootstrap sampling to measure the variance of that effect, again finding the effect using the non-recommended approach, which of these is closest to that variance?

In [127]:
effects = []

for i in range(10000):
    sample = hw2_2.sample(n=len(hw2_2), replace=True)
 
    mean_treated = sample[sample["X"] == 1]["Y"].mean()
    mean_control = sample[sample["X"] == 0]["Y"].mean()
 
    effects.append(mean_treated - mean_control)
 
bootstrap_var = np.var(effects, ddof=1)
print("bootstrap variance is", bootstrap_var)
 
# Check it against the textbook formula for a difference of two means.
treated_y = hw2_2[hw2_2["X"] == 1]["Y"]
control_y = hw2_2[hw2_2["X"] == 0]["Y"]
 
formula_var = treated_y.var() / len(treated_y) + control_y.var() / len(control_y)
print("formula variance is", round(formula_var, 5))


bootstrap variance is 0.03193267139017407
formula variance is 0.03188


if we ran a linear regression (with intercept) to measure the effect, which of these is closest to the skewness of the effect measured? (Look up skewness online. You can use scipy.stats.skew to compute the skewness of a list of numbers.) 

In [128]:
effects = []

for i in range(10000):
    sample = hw2_2.sample(n=len(hw2_2), replace=True)
 
    fit = smf.ols("Y ~ X + Z", data=sample).fit()
    effect_this_sample = fit.params["X"]
 
    effects.append(effect_this_sample)
 
print("mean effect is", np.mean(effects))
print("skewness is", skew(effects))


mean effect is 2.8199924633920093
skewness is 0.0481332948987788


### Week 3 ###

In [129]:
hw3_1 = pd.read_csv("homework_3.1.csv")
hw3a = pd.read_csv("homework_3.2.a.csv")
hw3b = pd.read_csv("homework_3.2.b.csv")

hw3_1["centered"] = hw3_1["time"] - 50
hw3_1["after"] = (hw3_1["time"] >= 50).astype(int)

Which dataset is most likely to have a discontinuity (or has the strongest discontinuity) in the value at the event (at time = 50)? 

In [130]:
model_value1 = smf.ols("value1 ~ centered + after", data=hw3_1).fit()
model_value2 = smf.ols("value2 ~ centered + after", data=hw3_1).fit()
model_value3 = smf.ols("value3 ~ centered + after", data=hw3_1).fit()
 
jump_value1 = model_value1.params["after"]
jump_value2 = model_value2.params["after"]
jump_value3 = model_value3.params["after"]
 
print("value1 jump =", jump_value1, "t =", model_value1.tvalues["after"])
print("value2 jump =", jump_value2,"t =", model_value2.tvalues["after"])
print("value3 jump =", jump_value3,"t =", model_value3.tvalues["after"])


value1 jump = 0.8508131385897626 t = 1.7367453377236965
value2 jump = 0.6827457703127844 t = 1.598863621390551
value3 jump = 1.7672540158368797 t = 4.356867546694259


Which dataset is most likely to have a discontinuity (or has the strongest discontinuity) in the derivative (at time = 50)? 

In [131]:
model_value1 = smf.ols("value1 ~ centered * after", data=hw3_1).fit()
model_value2 = smf.ols("value2 ~ centered * after", data=hw3_1).fit()
model_value3 = smf.ols("value3 ~ centered * after", data=hw3_1).fit()

slope_change_value1 = model_value1.params["centered:after"]
slope_change_value2 = model_value2.params["centered:after"]
slope_change_value3 = model_value3.params["centered:after"]
 
print("value1 slope difference =", slope_change_value1,"t =", model_value1.tvalues["centered:after"])
print("value2 slope difference =", slope_change_value2,"t =", model_value2.tvalues["centered:after"])
print("value3 slope difference =", slope_change_value3,"t =", model_value3.tvalues["centered:after"])

value1 slope difference = 0.10532461852916128 t = 7.951258658071948
value2 slope difference = 0.0369213706238385 t = 2.566732049342885
value3 slope difference = 0.05069507670249036 t = 3.857133079449938


Which dataset likely has the largest treatment effect, assuming that the treatment and control groups have parallel trends? 

In [132]:
model_group1 = smf.ols("outcome1 ~ group1 * time1", data=hw3a).fit()
model_group2 = smf.ols("outcome2 ~ group2 * time2", data=hw3b).fit()
 
effect_group1 = model_group1.params["group1:time1"]
effect_group2 = model_group2.params["group2:time2"]
 
print(effect_group1)
print(effect_group2)


0.6858469689823647
1.349858924679679


Using the standard errors for regression, which dataset has the most statistically significant (and nonzero) treatment effect? 

In [133]:
stderror_group1 = model_group1.bse["group1:time1"]
stderror_group2 = model_group2.bse["group2:time2"]
 
t_group1 = model_group1.tvalues["group1:time1"]
t_group2 = model_group2.tvalues["group2:time2"]
 
print(effect_group1, stderror_group1, t_group1)
print(effect_group2, stderror_group2, t_group2)


0.6858469689823647 0.0625224527016276 10.969610745365252
1.349858924679679 0.1470470684854061 9.179774466660977


Which of these is closest to the treatment effect for group 2? 

In [134]:
model_group2 = smf.ols("outcome2 ~ group2 * time2", data=hw3b).fit()
effect_group2 = model_group2.params["group2:time2"]
 
print(effect_group2)
print()
 
# The same number worked out from the four group averages.
control_before = hw3b[(hw3b["group2"] == 0) & (hw3b["time2"] == 0)]["outcome2"].mean()
control_after  = hw3b[(hw3b["group2"] == 0) & (hw3b["time2"] == 1)]["outcome2"].mean()
treated_before = hw3b[(hw3b["group2"] == 1) & (hw3b["time2"] == 0)]["outcome2"].mean()
treated_after  = hw3b[(hw3b["group2"] == 1) & (hw3b["time2"] == 1)]["outcome2"].mean()
 
print(control_before, control_after)
print(treated_before, treated_after)
print()
 
control_change = control_after - control_before
treated_change = treated_after - treated_before
 
print(control_change)
print(treated_change)
print(treated_change - control_change)


1.349858924679679

0.1021073578290933 1.367620588471783
1.9498260433077885 4.565198198630155

1.2655132306426897
2.6153721553223663
1.3498589246796766


### Week 4 ###

In [135]:
hw4_1 = pd.read_csv("homework_4.1 (1).csv")

hw4a = pd.read_csv("homework_4.2.a (1).csv")
hw4a = hw4a.rename(columns={"X": "score", "Y": "admit"})

hw4b = pd.read_csv("homework_4.2.b (1).csv")
hw4b= hw4b.rename(columns={"X2": "score", "Y2": "admit"})

hw4a["centered"] = hw4a["score"] - 80
hw4a["after"] = (hw4a["score"] >= 80).astype(int)
 
hw4b["centered"] = hw4b["score"] - 80
hw4b["after"] = (hw4b["score"] >= 80).astype(int)


As in the explanation, try averaging the ﻿Y﻿ difference and ﻿X﻿ difference (over ﻿W﻿ and ﻿Z﻿) in two ways: 

Subtract the average ﻿Y﻿ value for ﻿Z equals 1﻿and ﻿Z equals 0﻿. Subtract the average ﻿X﻿value for ﻿Z equals 1﻿ and ﻿Z equals 0﻿. Divide the two. 
Find the average ﻿Y﻿ value for ﻿Z equals 1﻿and ﻿Z equals 0﻿ for a narrow range of ﻿W﻿. Find the average ﻿X﻿value for ﻿Z equals 1﻿ and ﻿Z equals 0﻿ for the same narrow range of ﻿W﻿. Take the ratio to find the effect. Then average this over all the ranges of ﻿W﻿.

In [136]:
mean_y_x1 = hw4_1[hw4_1["Z"] == 1]["Y"].mean()
mean_y_z0 = hw4_1[hw4_1["Z"] == 0]["Y"].mean()
mean_x_z1 = hw4_1[hw4_1["Z"] == 1]["X"].mean()
mean_x_z0 = hw4_1[hw4_1["Z"] == 0]["X"].mean()

change_y = mean_y_x1 - mean_y_z0
change_x = mean_x_z1 - mean_x_z0

print("change Y is", change_y)
print("change X is", change_x)
print("mtd 1 is", change_y / change_x)
print()

hw4_1["W_"] = pd.qcut(hw4_1["W"], 10)

ratio = []
for name, rows in hw4_1.groupby("W_", observed=True):
    y_z1 = rows[rows["Z"] == 1]["Y"].mean()
    y_z0 = rows[rows["Z"] == 0]["Y"].mean()
    x_z1 = rows[rows["Z"] == 1]["X"].mean()
    x_z0 = rows[rows["Z"] == 0]["X"].mean()

    ratio.append((y_z1 - y_z0) / (x_z1 - x_z0))

print(len(ratio))
print("mtd 2 is", sum(ratio) / len(ratio))

change Y is 1.5908452261753534
change X is 1.0185589891466347
mtd 1 is 1.5618587073765746

10
mtd 2 is 1.5088963715223709


Did you need to know W to do this? 

In [137]:
change_y = hw4_1[hw4_1["Z"] == 1]["Y"].mean() - hw4_1[hw4_1["Z"] == 0]["Y"].mean()
change_x = hw4_1[hw4_1["Z"] == 1]["X"].mean() - hw4_1[hw4_1["Z"] == 0]["X"].mean()
instrument_effect = change_y / change_x

model_x_only = smf.ols("Y ~ X", data=hw4_1).fit()
model_x_and_w = smf.ols("Y ~ X + W", data=hw4_1).fit()

effect_x_only = model_x_only.params["X"]
effect_x_and_w = model_x_and_w.params["X"]

stage1 = smf.ols("X ~ Z", data=hw4_1).fit()
hw4_1["X_predicted"] = stage1.fittedvalues
stage2 = smf.ols("Y ~ X_predicted", data=hw4_1).fit()
effect_two_stage = stage2.params["X_predicted"]

print(instrument_effect)
print(effect_x_only)
print(effect_x_and_w)
print(effect_two_stage)

1.5618587073765746
1.5109859346672123
1.4954320196449116
1.5618587073765773


Given a cutoff score of 80, which dataset seems most likely to involve a nonzero linear term, allowing ﻿Y﻿to relate linearly to ﻿X﻿ before and after the cutoff? 

In [138]:
fit_a = smf.ols("admit ~ centered * after", data=hw4a).fit()
fit_b = smf.ols("admit ~ centered * after", data=hw4b).fit()

slope_a = fit_a.params["centered"]
slope_b = fit_b.params["centered"]

t_a = fit_a.tvalues["centered"]
t_b = fit_b.tvalues["centered"]

p_before_a = fit_a.pvalues["centered"]
p_before_b = fit_b.pvalues["centered"]

print(slope_a)
print(t_a)
print(p_before_a)
print(slope_b)
print(t_b)
print(p_before_b)

0.00022383079714033787
0.6351812090522163
0.525311690630157
0.010216711274400336
36.32437572701721
5.022860242370179e-287


In that dataset, is ﻿Y﻿ increasing or decreasing before the cutoff? 

In [140]:
fit_b = smf.ols("admit ~ centered * after", data=hw4b).fit()
slope_before = fit_b.params["centered"]
 
print("dataset b, slope before the cutoff =", round(slope_before, 5))
 
if slope_before > 0:
    print("Positive, Y is increasing before the cutoff")
else:
    print("Negative, Y is decreasing before the cutoff")


dataset b, slope before the cutoff = 0.01022
Positive, Y is increasing before the cutoff


Is ﻿Y﻿'s slope higher or lower after the cutoff compared with before? 

In [142]:
fit_b = smf.ols("admit ~ centered * after", data=hw4b).fit()
 
slope_before = fit_b.params["centered"]
change_at_cutoff = fit_b.params["centered:after"]
slope_after = slope_before + change_at_cutoff
 
print("slope before =", round(slope_before, 5))
print("change at the cutoff =", round(change_at_cutoff, 5),
      " t =", round(fit_b.tvalues["centered:after"], 2))
print("slope after  =", round(slope_after, 5))
 
if slope_after < slope_before:
    print("lower after the cutoff")
else:
    print("higher after the cutoff")


slope before = 0.01022
change at the cutoff = -0.00521  t = -12.53
slope after  = 0.00501
lower after the cutoff
